# 🚀 Notebook 05 — Model Deployment & Batch Scoring

**Goal:** Register the best model in MLflow Model Registry and batch score all 500 accounts. Results saved to `gold_credit_risk_scores` Delta Table.

> **Run time:** ~5 min

In [ ]:
# Import MLflow registry utilities and preprocessing helpers for model deployment
import mlflow
import mlflow.sklearn
from mlflow import MlflowClient
import pandas as pd
import numpy as np
from sklearn.preprocessing import LabelEncoder

# Initialize the MLflow client and define the registry model name used for deployment
client = MlflowClient()
model_name = 'CreditRiskScorer'

# Find the best run by AUC-ROC
# Find the best experiment run by AUC-ROC so it can be promoted to production
runs = mlflow.search_runs(
    experiment_names=['CreditRiskScoring'],
    order_by=['metrics.auc_roc DESC']
)
best_run = runs.iloc[0]
# Display the winning run that will be registered and deployed
print(f'Best model: {best_run["tags.mlflow.runName"]}')
print(f'AUC-ROC:    {best_run["metrics.auc_roc"]:.4f}')
print(f'Run ID:     {best_run["run_id"]}')


## Step 1 — Register Model in MLflow Model Registry

In [ ]:
# Register the best model
# Build the MLflow model URI for the best-performing training run
model_uri = f"runs:/{best_run['run_id']}/{best_run['tags.mlflow.runName'].lower().replace(' ','_')}"

# Register the winning model in the MLflow Model Registry
registered = mlflow.register_model(model_uri=model_uri, name=model_name)
print(f'\n✅ Registered model: {registered.name} v{registered.version}')

# Transition to Production
# Promote the newly registered model version to Production and archive older versions
client.transition_model_version_stage(
    name=model_name,
    version=registered.version,
    stage='Production',
    archive_existing_versions=True
)
# Confirm the production stage transition completed successfully
print(f'✅ Model v{registered.version} promoted to Production')


## Step 2 — Load Production Model

In [ ]:
# Load directly from Model Registry
# Load the production model back from the registry to verify it is deployable
model = mlflow.pyfunc.load_model(f'models:/{model_name}/Production')
# Confirm the production model can be resolved from the registry
print(f'Loaded: {model_name} (Production)')


## Step 3 — Batch Score All 500 Accounts

In [ ]:
# Prepare full dataset for scoring
# Load the full engineered feature store so every account can be batch scored
df_all = spark.table('silver_credit_risk_features').toPandas()

# Encode the categorical deployment features to match the training feature space
cat_cols = ['AccountType','Branch','Status','LoanPurpose','EmploymentType','HomeOwnership']
for col in cat_cols:
    le = LabelEncoder()
    df_all[col + '_enc'] = le.fit_transform(df_all[col].astype(str))

# Recreate the exact feature list expected by the production model
feature_cols = [
    'CreditScore','AnnualIncome','LoanAmount','EmploymentYears',
    'DebtToIncomeRatio','NumOpenAccounts','NumDelinquencies',
    'MonthsSinceLastDelinquency','LoanTermMonths',
    'TotalTransactions','TotalLoanAmount','AvgTransactionAmount',
    'NumLoans','MaxTransactionAmount',
    'AccountType_enc','LoanPurpose_enc','EmploymentType_enc','HomeOwnership_enc'
]
# Build the batch scoring matrix and replace any missing values with zero
X_all = df_all[feature_cols].fillna(0)

# Score
# Generate default probability scores for every account using the production model
scores = model.predict(X_all)
# Review the number of scored accounts and the score distribution summary
print(f'Scored {len(scores)} accounts')
print(f'Score distribution: min={scores.min():.3f} mean={scores.mean():.3f} max={scores.max():.3f}')


## Step 4 — Save Scores to Gold Table

In [ ]:
# Import pandas and datetime helpers for building the scored output table
import pandas as pd
from datetime import datetime

# Build output dataframe
# Assemble scoring results with probabilities, risk bands, timestamps, and model metadata
df_scores = pd.DataFrame({
    'AccountID':        df_all['AccountID'],
    'DefaultProbability': np.round(scores, 4),
    'RiskCategory': pd.cut(
        scores,
        bins=[0, 0.2, 0.4, 0.6, 1.0],
        labels=['Low Risk','Medium Risk','High Risk','Very High Risk']
    ).astype(str),
    'ActualDefault': df_all['IsDefault'],
    'ScoredAt':      datetime.utcnow().strftime('%Y-%m-%dT%H:%M:%SZ'),
    'ModelVersion':  f'v{registered.version}',
    'ModelName':     model_name
})

# Save to Gold Delta Table
# Persist the scored accounts to the gold Delta table used for monitoring and reporting
spark.createDataFrame(df_scores) \
     .write.format('delta').mode('overwrite') \
     .saveAsTable('gold_credit_risk_scores')

# Confirm the scoring table was written and review the risk-category mix
print('✅ gold_credit_risk_scores saved')
print('\nRisk Category Distribution:')
print(df_scores['RiskCategory'].value_counts().sort_index())


## Step 5 — Query Gold Table

In [ ]:
%%sql
-- Summarize the gold scoring table by risk category to review portfolio mix and realized defaults
SELECT
    RiskCategory,
    COUNT(*) AS Accounts,
    ROUND(AVG(DefaultProbability) * 100, 1) AS AvgDefaultProb_Pct,
    SUM(ActualDefault) AS ActualDefaults
FROM gold_credit_risk_scores
GROUP BY RiskCategory
ORDER BY AvgDefaultProb_Pct DESC
